## 最初の大きなプロジェクト - デジタルツイン

### まずは、Pushoverの紹介から

Pushoverは、スマートフォンにプッシュ通知を送るための便利なツールです。

セットアップとインストールはとても簡単です！

https://pushover.net/ にアクセスして、右上の「Login or Signup」をクリックし、無料アカウントを作成してAPIキーを発行してください。

登録が完了したら、ホーム画面で「Create an Application/API Token」をクリックし、任意の名前（例：Agents）を付けて「Create Application」をクリックします。

その後、`.env`ファイルに以下の2行を追加します：

PUSHOVER_USER=_Pushoverのホーム画面右上にあるキーを入力します。おそらく「u」で始まります_
PUSHOVER_TOKEN=_「Agents」など、新しく作成したアプリケーションをクリックした先にあるキーを入力します。おそらく「a」で始まります_

`.env`ファイルを保存し忘れないようにしてください。そして保存後に`load_dotenv(override=True)`を実行して、環境変数を設定してください。

最後に、「Add Phone, Tablet or Desktop」をクリックして、スマートフォンにインストールしてください。

## ご注意 - 動画からの変更点

動画では、ツインをHuggingFace Spacesに無料でデプロイしています。しかし、HuggingFaceは最近、これを無料でサポートしなくなりました！

無料の代替方法がありますので、このラボの後半で説明し、手順を示します。

In [ ]:
# インポート

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [ ]:
# いつもの開始処理

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# Pushover用

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [ ]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [ ]:
tools

In [ ]:
# この関数はツール呼び出しのリストを受け取り、それらを実行します。これがIF文です！！

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # 大きなIF文です！！！

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

## Pythonの組み込み関数globals()を使う

Pythonには、すべてのグローバル関数へアクセスできる辞書があります。

補足：デプロイする際には、当然これをもっと保護された形で使うようにします……

In [ ]:
globals()["record_unknown_question"]("this is a really hard question")

In [ ]:
# これにより、IF文を避けた、より洗練された方法が実現します。

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Pythonモジュールへの変換

ラボのコードをPythonモジュールに変換しました。ノートブックでの実験が終わった後に行うと良い、優れたプラクティスです。

上記のコードをすべて1つのPythonスクリプトにまとめることもできます。しかし、関心事ごとに異なるモジュールへコードを整理するほうが望ましく、私はそのようにしました。

`context.py`は静的データを読み込み、システムプロンプトを構築します。

`tools.py`には、ツールとその関連JSONを管理・呼び出すためのコードがすべて含まれています。

`app.py`には、GradioアプリとOpenAIの呼び出しが含まれています。

`styles.py`には、Gradioに適用するスタイルが含まれており、これは全面的にClaude Codeによって書かれました！

まずは自分で挑戦してみて、その後私のバージョンと比較してみるといいでしょう。

試してみるには、Cursorでターミナルを開いてください：

`cd 1_foundations`
`cd twin`
`uv run app.py`

# 緊急連絡！ご注意ください……

2026年7月9日をもって、HuggingFaceは突然、GradioアプリをHuggingFace Spacesに無料でデプロイすることを許可しなくなりました。

これはかなり嫌なサプライズです！

この決定が撤回されることを期待していますが、それまでの間、無料の代替手段をご紹介します：Renderを使う方法です。

完全な手順は[このディレクトリ内のRENDER_INSTRUCTIONSファイル](RENDER_INSTRUCTIONS.md)にあります。

HuggingFaceへの支払いを気にしないのであれば、以下に元の手順を残しています。

また、私のデジタルツインについての手順もあります。こちらはfly.io上で非常に低コストで動作しています。
https://edwarddonner.com/avatar

私のツインでは、Pushで私に通知を送るだけでなく、本物の私とチャットすることもできます！これをどうやって作ったかの動画と、自分でも作りたい場合の手順を用意しています。私はこのCareer Conversationsアプリから始めました。
https://youtu.be/srlhW4H-Gtg

## HF Spacesを使った元の手順（現在は無料ではありません）

HuggingFace Spacesにデプロイします。

始める前に：`twin`ディレクトリ内のファイル - あなたのLinkedInプロフィールと summary.txt - を、あなた自身について語る内容に更新しておいてください！

また、`twin`ディレクトリ内にREADMEファイルが存在しないことも確認してください。存在する場合は削除してください。デプロイの過程で、このディレクトリに新しいREADMEファイルが自動的に作成されます。

## デプロイ パート1：HuggingFace

1. https://huggingface.co にアクセスし、アカウントを作成します
2. 右上のAvatarメニューから「Access Tokens」を選びます。「Create New Token」を選択します。WRITE権限を付与してください - WRITE権限が必要です！新しいキーは記録しておいてください。
3. Cursorのターミナルで、`uvx hf auth login --token YOUR_TOKEN_HERE`（例：`uvx hf auth login --token hf_xxxxxx`）を実行し、キーを使ってコマンドラインからログインします。その後、`uvx hf auth whoami`を実行して、ログインできているか確認します
4. 新しいトークンを、今後のために.envファイルに追加します：`HF_TOKEN=hf_xxx`

## デプロイ パート2：Push！

1. twinディレクトリに移動します：`cd 1_foundations` の後 `cd twin`
2. twinディレクトリから、次を入力します：`uv run gradio deploy`
3. デフォルト値を選びながら、案内に従ってください：名前は`twin`とし、app.pyを指定し、ハードウェアはcpu-basicを選び、secretsの入力が必要かの質問には「No」、github actionsについても「no」と答えます。

### デプロイ パート3：Secrets

1. https://huggingface.co にアクセスし、Avatarをクリックしてプロフィールに移動し、Spaceを選択します
2. 3点メニューから「Settings」を選びます
3. 「Variables and Secrets」セクションまでスクロールします
4. 「New Secret」（「New Variable」ではありません）を押し、名前として`OPENAI_API_KEY`、値としてご自身の.envファイルにあるキー（またはお使いのLLMに対応するキー）を入力します。ここは正確に入力するよう注意してください！
5. .envファイルの`PUSHOVER_USER`と`PUSHOVER_TOKEN`についても同様に行います
6. Settingsの上部近くにある「Restart space」をクリックして再起動します
7. 上部近くの「App」をクリックしてアプリに戻り、再起動が終わったら - お楽しみください！

### 他のサイトへの埋め込み

これを他のウェブサイトに埋め込むには、3点メニューから「Embed this space」を選択してください。

### トラブルシューティング

gradioのエラーが出た場合は、ログを開いてみてください（3点メニューの隣にあるボタンです）。
特にキーに関するデバッグ情報をもっと追加してみてください。

### スペースの再デプロイ

twinディレクトリから`uv run gradio deploy`を実行するだけです。もう一度スペースに名前を付けたい場合は、Gradioがそこに作成したREADME.mdファイルを削除する必要があるかもしれません。

### スペースの削除

3点メニューからSettings画面を選択すると、下部にDeleteオプションがあります。

デプロイについての詳しい情報：

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

### 私のデジタルツイン

そこで、私は自分のデジタルツインをさらに一段上のレベルに引き上げるための時間を使いました！
こちらです：
https://edwarddonner.com/avatar

Pushで私に通知を送るだけでなく、本物の私とチャットすることもできます！これをどうやって作ったかの動画と、自分でも作りたい場合の手順を用意しています。私はこのCareer Conversationsアプリから始めました。
https://youtu.be/srlhW4H-Gtg


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">・まず何よりも、これを自分自身のためにデプロイしてみましょう！これは実際に価値のあるツールです - 未来の履歴書です……<br/>
            ・次に、リソースを改善しましょう - 自分自身についてのより良いコンテキストを追加してください。RAGを知っているなら、自分についての知識ベースを追加してみましょう。<br/>
            ・さらにツールを追加してみましょう！LLMが読み書きできる、よくある質問と回答のSQLデータベースを持たせるのはどうでしょうか？<br/>
            ・4日目の演習からEvaluatorを取り入れ、他のエージェンティックパターンも追加してみましょう。<br/>
            ・一部の受講者は、Telegram連携を追加し、自分のサイトでツインと一緒に、訪問者とリアルタイムでチャットできるようにしています！
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">商業的な意義</h2>
            <span style="color:#00bfff;">明らかな用途（未来の履歴書）以外にも、これはドメインの専門知識を持ち、実世界と相互作用できるAIアシスタントが必要な、あらゆる状況でビジネス上の応用があります。
            </span>
        </td>
    </tr>
</table>